# NB12 — Lasso Regression (L1 Regularisation)

> **StatQuest: "Lasso has a sharp corner at zero, so some coefficients land exactly on it — those features are dropped."**

---

## The main ideas:

1. Lasso minimises: **SSR + lambda * sum(|b_j|)**
2. The L1 penalty (absolute value) has a **diamond shape** in 2D with corners on the axes
3. The OLS solution often "hits" one of those corners -> coefficient = exactly 0 -> feature eliminated
4. Ridge uses a circle (no corners) -> rarely gets exactly zero
5. Lasso = automatic feature selection


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

def flow_diagram(steps, title, colors=None, notes=None, figsize=(14, 2.8)):
    n = len(steps)
    default_colors = ['#1565C0','#2E7D32','#E65100','#6A1B9A',
                      '#00695C','#AD1457','#37474F','#4E342E',
                      '#0277BD','#558B2F','#C62828','#F57F17']
    colors = (colors or default_colors)[:n]
    notes  = notes or ['']*n
    fig, ax = plt.subplots(figsize=figsize)
    ax.set_xlim(-0.3, n*3.1); ax.set_ylim(-1.2, 2.4); ax.axis('off')
    bw, bh = 2.6, 1.3
    for i,(step,color,note) in enumerate(zip(steps,colors,notes)):
        x = i*3.1
        box = FancyBboxPatch((x,0.2),bw,bh,boxstyle="round,pad=0.12",
                             facecolor=color,edgecolor='white',linewidth=1.5,alpha=0.90)
        ax.add_patch(box)
        ax.text(x+bw/2,0.2+bh/2,step,ha='center',va='center',fontsize=8.5,
                color='white',fontweight='bold',multialignment='center')
        if note:
            ax.text(x+bw/2,0.02,note,ha='center',va='top',fontsize=7,
                    color='#555',style='italic')
        if i < n-1:
            ax.annotate('',xy=(x+bw+0.38,0.2+bh/2),xytext=(x+bw+0.08,0.2+bh/2),
                       arrowprops=dict(arrowstyle='->',color='#444',lw=2.2))
    ax.set_title(title,fontsize=11,fontweight='bold',pad=6,color='#222')
    plt.tight_layout(pad=0.4); plt.show()

flow_diagram(
    steps=[
        'Many features\n(some useless)',
        'Lasso adds\nL1 penalty:\nSSR + lambda*sum(|b|)',
        'Geometric\nview: L1 ball\nhas CORNERS',
        'OLS hits\ncorner -> b_j=0\nfeature dropped',
        'Tune lambda\nwith LassoCV',
        'Only selected\nfeatures\nremain',
        'Interpret\nremaining\ncoefficients',
    ],
    title='NB12 Conceptual Flow: Lasso Regression (L1) for Feature Selection',
    colors=['#37474F','#C62828','#1565C0','#2E7D32','#E65100','#6A1B9A','#00695C'],
    figsize=(16, 2.8),
)


## Geometric intuition: why L1 creates sparsity

In 2D, the constraint region for:
- **Ridge (L2):** a circle — smooth, no special points
- **Lasso (L1):** a diamond — 4 corners, each ON an axis (b_j = 0)

The OLS objective (elliptical contours) moves toward the minimum until it first touches the constraint region.
**A circle has no corners — it rarely hits an axis.**
**A diamond has corners ON the axes — the first contact is often at a corner.**
Corner contact = one coefficient exactly zero = that feature eliminated.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

theta = np.linspace(0, 2*np.pi, 500)
fig, axes = plt.subplots(1, 2, figsize=(11, 5))
true_ols = np.array([1.4, 0.9])

for ax, title, is_lasso in zip(axes,
    ['Ridge (L2): circle, no corners', 'Lasso (L1): diamond, corners on axes'],
    [False, True]):

    ax.set_xlim(-2.2, 2.2); ax.set_ylim(-2.2, 2.2); ax.set_aspect('equal')
    ax.axhline(0, color='k', linewidth=0.5); ax.axvline(0, color='k', linewidth=0.5)
    ax.set_xlabel('b1'); ax.set_ylabel('b2')

    if is_lasso:
        diamond = np.array([[1,0],[-1,0],[0,-1],[0,1],[1,0]])
        ax.fill(diamond[:,0], diamond[:,1], alpha=0.2, color='steelblue')
        ax.plot(diamond[:,0], diamond[:,1], 'b-', linewidth=2.5, label='L1 ball')
        constrained = np.array([1, 0])
        ax.plot(*constrained, 'go', markersize=12, zorder=5, label='Lasso solution (b2=0!)')
    else:
        ax.fill(np.cos(theta), np.sin(theta), alpha=0.2, color='steelblue')
        ax.plot(np.cos(theta), np.sin(theta), 'b-', linewidth=2.5, label='L2 ball')
        ax.plot(0.77, 0.64, 'go', markersize=12, zorder=5, label='Ridge solution')

    for r in [0.5, 0.9, 1.4, 2.0]:
        cx = true_ols[0] + r*np.cos(theta)
        cy = true_ols[1] + 0.6*r*np.sin(theta)
        ax.plot(cx, cy, 'r-', alpha=0.35, linewidth=1)

    ax.plot(*true_ols, 'r*', markersize=16, label='OLS solution (unconstrained)')
    ax.set_title(title); ax.legend(fontsize=8); ax.grid(alpha=0.3)

plt.suptitle('Why Lasso performs variable selection: the diamond has corners on the axes',
             fontsize=11, fontweight='bold')
plt.tight_layout(); plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import Lasso, LassoCV
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import fetch_california_housing

housing = fetch_california_housing()
X_std   = StandardScaler().fit_transform(housing.data)
y       = housing.target
names   = list(housing.feature_names)

# Regularisation path
lambdas = np.logspace(-4, 1, 200)
coefs   = np.array([Lasso(alpha=l, max_iter=10000).fit(X_std, y).coef_
                    for l in lambdas])

fig, ax = plt.subplots(figsize=(10, 5))
for j, name in enumerate(names):
    ax.semilogx(lambdas, coefs[:, j], linewidth=2, label=name)
ax.axhline(0, color='black', linewidth=1.5)
ax.set_xlabel('lambda (log scale)'); ax.set_ylabel('Coefficient value')
ax.set_title('Lasso path: coefficients HIT zero (hard cutoff) as lambda increases\n'
             '(Compare Ridge: they approach but never reach zero)')
ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

# Optimal lambda
lasso_cv = LassoCV(cv=5, max_iter=20000).fit(X_std, y)
print(f"Optimal lambda: {lasso_cv.alpha_:.6f}")
print(f"\nSelected features at optimal lambda:")
for name, coef in zip(names, lasso_cv.coef_):
    status = 'SELECTED' if coef != 0 else 'zeroed out'
    bar    = '#' * int(abs(coef)*20) if coef != 0 else ''
    print(f"  {name:<15} {coef:>8.4f}  {status}  {bar}")
print(f"\n{sum(lasso_cv.coef_ != 0)}/{len(names)} features kept")


## Ridge vs Lasso at a glance

| | Ridge | Lasso |
|--|-------|-------|
| Penalty | sum(b^2) | sum(|b|) |
| Shape of constraint | Circle (smooth) | Diamond (corners) |
| Variable selection | No — all kept | Yes — some b_j = 0 |
| Correlated features | Shares credit equally | Picks one, drops others |
| When to use | All features matter | Many features, few relevant |
| Interpretability | Moderate | High (sparse model) |

**Next: NB13 — ElasticNet combines both, getting the best of each.**
